# Tutorial: Cell-Level LDSC analysis

This tutorial demonstrates how to perform cell-type-specific LD Score regression (LDSC) analysis through the `cellink` package. The `cellink` package provides a unified interface to LDSC and its preparation scripts, making it easier to perform comprehensive genetic analyses that identify which cell types are most relevant to complex traits and diseases.

This notebook assumes familiarity with single-cell data processing and basic statistical genetics concepts. The `cellink` package provides convenient wrapper functions that handle data preparation and formatting for LDSC. For LDSC installation, please follow instructions [here](https://github.com/bulik/ldsc). We recommend utilizing LDSC via a Docker image like [this one](https://hub.docker.com/r/zijingliu/ldsc). For usage on HPCs, please consider using singularity or enroot.

## Environment Setup

We begin by importing necessary libraries and defining key parameters for our analysis. The `cellink` package provides wrapper functions for LDSC that automatically handle preprocessing, data formatting and preparation.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import os

from joblib import Parallel, delayed

from cellink.resources import get_dummy_onek1k
from cellink.tl.external import (
    preprocess_for_sldsc,
    generate_sldsc_genesets,
    generate_gene_coord_file,
    configure_ldsc_runner,
    make_annot_from_donor_data,
    make_annot_from_bimfile,
    munge_sumstats,
    estimate_heritability,
    estimate_genetic_correlation,
    compute_ld_scores_with_annotations_from_bimfile,
    estimate_celltype_specific_heritability,
)
from cellink.resources import get_1000genomes_ld_scores, get_1000genomes_ld_weights
from cellink.resources import get_gwas_catalog_study_summary_stats

# Analysis parameters
chrom = 22
cell_type = "CD8 Naive"
celltype_key = "predicted.celltype.l2"
original_donor_col = "donor_id"
# ldsc runs N_JOBS at a time, one core each: one numpy thread per ldsc process
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
N_JOBS = min(8, os.cpu_count())

site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
!pwd

/path/to/cellink/docs/tutorials


## Load and Prepare Data

We load the OneK1K dataset, which contains both genotype and single-cell expression data. We also add gene annotations from Ensembl for our analysis.

In [3]:
!pip install pybiomart

In [4]:
# Load the dummy OneK1K dataset (genotype + single-cell expression).
# Downloaded and cached automatically. See get_dummy_onek1k docs.
dd = get_dummy_onek1k()
print(f"Dataset shape: {dd.shape}")

[2026-11-14 23:56:41,968] INFO:root: ~/cellink_data/dummy_onek1k/dummy_onek1k_dataset.dd.h5 already exists
[2026-11-14 23:56:41,969] INFO:root: Verifying checksum
[2026-11-14 23:56:47,208] INFO:root: Loaded dummy OneK1K dataset: (300, 146939, 125913, 34830)
Dataset shape: (300, 146939, 125913, 34830)


In [5]:
dd.G.obs["donor_id"] = dd.G.obs.index

In [6]:
# The dummy dataset is already small, but this shows the subsampling pattern
# you'd use on the full OneK1K dataset (~1.25M cells) for per-cell-type
# specificity estimation.
np.random.seed(0)
n_cells_keep = 30_000
cell_idx = np.sort(np.random.choice(dd.shape[2], min(n_cells_keep, dd.shape[2]), replace=False))
dd = dd[:, :, cell_idx, :].copy()
print(f"Subsampled to {dd.shape[2]:,} cells")

Subsampled to 30,000 cells


### Cell-Type-Specific LDSC analysis

Cell-type-specific LDSC analysis helps identify which cell types are most relevant to complex traits by testing whether genetic variants associated with a trait are enriched in genes specifically expressed in certain cell types. This analysis follows the method described in [Duncan et al. 2025](https://www.nature.com/articles/s41593-024-01834-w) to identify specifically highly expressed genes per cell type.

#### Step 1: Preprocessing and Gene Set Generation
First, we preprocess the single-cell data to compute cell-type-specific gene expression and identify genes that are specifically expressed in each cell type.

In [7]:
dd.C.var["gene"] = dd.C.var_names
adata = dd.C.copy()
adata_filtered, mean_expr, specificity = preprocess_for_sldsc(
    adata,
    celltype_col=celltype_key,
    gene_identifier_mode="ensembl",
    gene_col="gene",
    genome_build="GRCh37",
    inplace=False,
)

# Backfill chromosome annotations onto dd.C.var so later steps can filter
# genes by chromosome the same way they filter SNPs (kept for parity with
# the full OneK1K dataset, which has no gene coordinates pre-attached).
dd.C.var["chrom"] = adata_filtered.var["chrom"].reindex(dd.C.var_names)

[2026-11-14 23:56:48,085] INFO:cellink.tl.external._sldsc_utils: Fetching gene annotations from GRCh37...
[2026-11-14 23:56:48,087] INFO:cellink.tl.external._biomart: Querying Ensembl BioMart at https://grch37.ensembl.org (hsapiens_gene_ensembl: hgnc_symbol, external_gene_name, ensembl_gene_id, chromosome_name, start_position, end_position, gene_biotype)
[2026-11-14 23:56:51,305] INFO:cellink.tl.external._sldsc_utils: Fetched annotations for 63763 genes from GRCh37
[2026-11-14 23:56:51,353] INFO:cellink.tl.external._sldsc_utils: Removing version suffixes from Gene IDs
[2026-11-14 23:56:51,668] INFO:cellink.tl.external._sldsc_utils: Dropping conflicting columns from adata.var before merge: ['chrom', 'start', 'end']
[2026-11-14 23:56:51,868] INFO:cellink.tl.external._sldsc_utils: Annotated 30793 / 34830 genes.
[2026-11-14 23:56:51,878] INFO:cellink.tl.external._sldsc_utils: Using annotation columns: gene=gene, biotype=gene_biotype, chr=chrom, start=start, end=end
[2026-11-14 23:56:51,879

Aggregating clusters: 100%|██████████| 31/31 [00:01<00:00, 23.07it/s]

[2026-11-14 23:56:53,790] INFO:cellink.tl.external._sldsc_utils: Log1p applied.
[2026-11-14 23:56:53,816] INFO:cellink.tl.external._sldsc_utils: Computing mean expression for predicted.celltype.l2


[2026-11-14 23:56:54,014] INFO:cellink.tl.external._sldsc_utils: Computing specificity scores
[2026-11-14 23:56:54,144] INFO:cellink.tl.external._sldsc_utils: Final data shape: (30000, 15074)
[2026-11-14 23:56:54,145] INFO:cellink.tl.external._sldsc_utils: Mean expression shape: (15074, 31)
[2026-11-14 23:56:54,146] INFO:cellink.tl.external._sldsc_utils: Specificity shape: (15074, 31)


Next, we generate gene sets for each cell type containing the top 10% (top_frac) most specifically expressed genes. These gene sets will be used to create genomic annotations.

In [8]:
summary = generate_sldsc_genesets(specificity, dd.C, out_dir="ldsc_genesets", top_frac=0.10, overwrite=True)
summary

[2026-11-14 23:56:54,166] INFO:cellink.tl.external._sldsc_utils: Writing gene sets to ldsc_genesets
[2026-11-14 23:56:54,172] INFO:cellink.tl.external._sldsc_utils: specificity_df index looks like Ensembl IDs; using them directly.
[2026-11-14 23:56:54,174] INFO:cellink.tl.external._sldsc_utils: Selecting top 1508 genes (10.0%) per cell type
[2026-11-14 23:56:54,656] INFO:cellink.tl.external._sldsc_utils: Wrote control gene set with 15074 genes
[2026-11-14 23:56:54,658] INFO:cellink.tl.external._sldsc_utils: Generated 31 cell-type-specific gene sets


,cell_type,n_genes,output_path
0,ASDC,1508,ldsc_genesets/ASDC.GeneSet
1,B intermediate,1508,ldsc_genesets/B_intermediate.GeneSet
2,B memory,1508,ldsc_genesets/B_memory.GeneSet
3,B naive,1508,ldsc_genesets/B_naive.GeneSet
4,CD14 Mono,1508,ldsc_genesets/CD14_Mono.GeneSet
5,CD16 Mono,1508,ldsc_genesets/CD16_Mono.GeneSet
6,CD4 CTL,1508,ldsc_genesets/CD4_CTL.GeneSet
7,CD4 Naive,1508,ldsc_genesets/CD4_Naive.GeneSet
8,CD4 Proliferating,1508,ldsc_genesets/CD4_Proliferating.GeneSet
9,CD4 TCM,1508,ldsc_genesets/CD4_TCM.GeneSet


We also need gene coordinate files that map genes to genomic positions, in the genome build of the genotypes they are matched against: GRCh38 for the OneK1K genotypes in `dd`, GRCh37 for the 1000 Genomes reference panel used below:

In [9]:
# GRCh37 for the 1000 Genomes reference panel, GRCh38 for the OneK1K genotypes in dd
generate_gene_coord_file("gene_coords.txt", gene_identifier_mode="ensembl", genome_build="GRCh37", overwrite=True)
generate_gene_coord_file(
    "gene_coords_grch38.txt", gene_identifier_mode="ensembl", genome_build="GRCh38", overwrite=True
)

[2026-11-14 23:56:54,685] INFO:cellink.tl.external._sldsc_utils: Fetching gene annotations from Ensembl GRCh37...
[2026-11-14 23:56:54,686] INFO:cellink.tl.external._sldsc_utils: Fetching gene annotations from GRCh37...
[2026-11-14 23:56:54,687] INFO:cellink.tl.external._biomart: Querying Ensembl BioMart at https://grch37.ensembl.org (hsapiens_gene_ensembl: hgnc_symbol, external_gene_name, ensembl_gene_id, chromosome_name, start_position, end_position, gene_biotype)
[2026-11-14 23:56:54,978] INFO:cellink.tl.external._sldsc_utils: Fetched annotations for 63763 genes from GRCh37
[2026-11-14 23:56:54,980] INFO:cellink.tl.external._sldsc_utils: Removing version suffixes from gene identifiers
[2026-11-14 23:56:55,021] WARNING:cellink.tl.external._sldsc_utils: Removed 86 duplicate gene entries
[2026-11-14 23:56:55,032] INFO:cellink.tl.external._sldsc_utils: Writing 63677 gene coordinates to gene_coords.txt
[2026-11-14 23:56:55,132] INFO:cellink.tl.external._sldsc_utils: Successfully created 

#### Step 2: Configure LDSC Runner
Before running LDSC commands, we need to configure the runner. LDSC can be executed via a local installation or through container solutions like Docker or Singularity. Sample configuration files are provided in `./src/cellink/tl/external/config/` for local execution, Docker, and Singularity.

In [10]:
runner = configure_ldsc_runner("ldsc_singularity")

#### Step 3: Prepare Data for Analysis
To speed up the illustrative annotation of `dd`'s own genotypes below, we keep a random 0.1% of its SNPs per chromosome (at least one per chromosome). In a real analysis you would not subset the SNPs. The LD scores used downstream are computed from the full 1000 Genomes panel for every chromosome, since `ldsc.py --h2-cts` requires LD scores for the whole genome.


In [11]:
np.random.seed(42)
all_selected_idx = []
for c in range(1, 23):
    chrom_idx = np.where(dd.G.var.chrom == str(c))[0]
    n_snps = max(1, int(len(chrom_idx) * 0.001))
    selected_idx = np.random.choice(chrom_idx, n_snps, replace=False)
    all_selected_idx.extend(selected_idx)
all_selected_idx = np.sort(all_selected_idx)

dd = dd[:, all_selected_idx, :, :].copy()
dd

╔═ DonorData(n_donors=300, n_cells_per_donor=[77-131], donor_id='donor_id') ══════════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (donors)                                         ┃ C (cells)                                          ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ AnnData object with n_obs × n_vars = 300 × 157     │ AnnData object with n_obs × n_vars = 30,000 ×      │ ║
║ │                                                    │ 34,830                                             │ ║
║ │     obs: 'donor_id'                                │     obs: 'orig.ident', 'nCount_RNA',               │ ║
║ │                                                    │ 'nFeature_RNA', 'percent.mt', 'donor_id',          │ ║
║ │                                                    │ 'pool_number', 'predicted.celltype.l2',        

#### Step 4: Create Cell-Type Annotations

We create annotation files that mark which SNPs are near cell-type-specific genes. `cellink` provides two annotation types that produce identical output formats and are fully interchangeable downstream:

- **Binary** (`make_annot_from_donor_data` / `make_annot_from_bimfile`): SNPs within the window of a top-10% gene get `ANNOT=1`, all others `0`. Requires a `.GeneSet` file.
- **Continuous** (`make_annot_from_donor_data` / `make_annot_from_bimfile`): each SNP gets the specificity score of the highest-scoring gene whose window overlaps it (or `0` if none). Preserves the full quantitative signal.

**When to use which:** Binary annotations are the standard, widely-used approach (e.g. Finucane et al. LDSC-SEG), simple to interpret, easy to compare across studies, and the right default when you just want to test whether heritability is enriched in the top N% most cell-type-specific genes. Continuous annotations avoid the arbitrary top-N% cutoff and can be more powerful when specificity varies smoothly across genes rather than splitting cleanly into high/low; they are most useful when you already have a continuous score (e.g. from SEISMIC or Duncan et al.) that you don't want to threshold away. **This tutorial uses binary annotations as the main, recommended workflow**; the continuous variant is shown afterward as an alternative for when a continuous score is preferable.

Both write `CHR / BP / SNP / CM / ANNOT` files, so the downstream LD-score call is identical for both.

**Binary annotations from your own genotype data** (top-10% specificity genes → 1), using `make_annot_from_donor_data`, illustrative only; we don't carry this particular version forward (see why below):

In [12]:
for chrom_i in range(1, 23):
    for cell_type in ["CD8 Naive", "CD4 Naive"]:
        annot_file = f"{cell_type.replace(' ', '_')}_{chrom_i}.annot.gz"
        if os.path.exists(annot_file):
            continue
        dd_chrom = dd.sel(G_var=dd.G.var.chrom == str(chrom_i), C_var=dd.C.var.chrom == str(chrom_i)).copy()
        result = make_annot_from_donor_data(
            dd=dd_chrom,
            annot_file=annot_file,
            gene_set_file=f"./ldsc_genesets/{cell_type.replace(' ', '_')}.GeneSet",
            gene_coord_file="gene_coords_grch38.txt",
            gene_coord_genome_build="GRCh38",
            bim_genome_build="GRCh38",  # OneK1K genotypes
            windowsize=100000,
            runner=runner,
        )

[2026-11-14 23:57:17,480] INFO:cellink.tl.external._ldsc: Exporting genotype data to PLINK format for annotation creation
[2026-11-14 23:57:17,518] INFO:cellink.io._export: Wrote PLINK 1 binary files ldsc_annot.bed/.bim/.fam (300 x 1)
[2026-11-14 23:57:17,519] INFO:cellink.tl.external._ldsc: Creating annotation file: /ldsc/make_annot.py --bimfile ldsc_annot.bim --annot-file CD8_Naive_1.annot.gz --gene-set-file ./ldsc_genesets/CD8_Naive.GeneSet --gene-coord-file gene_coords_grch38.txt --windowsize 100000
[2026-11-14 23:57:17,524] INFO:cellink.tl._runner: Executing: singularity exec -B .:/data -B ~/cellink_data:/cellink_data --pwd /data ldsc.sif /ldsc/make_annot.py --bimfile /data/ldsc_annot.bim --annot-file /data/CD8_Naive_1.annot.gz --gene-set-file /data/ldsc_genesets/CD8_Naive.GeneSet --gene-coord-file /data/gene_coords_grch38.txt --windowsize 100000
[2026-11-14 23:57:20,870] INFO:cellink.tl._runner: making gene set bed file
making annot file

[2026-11-14 23:57:20,889] INFO:cellink.tl

**Why we switch to a reference panel for the rest of this tutorial:** OneK1K's own genotype SNP IDs are positional (e.g. `2_178356_A_C`), but real GWAS summary statistics are keyed by rsID. LD scores computed from `dd`'s own genotypes can never be merged with a real GWAS downstream. This is also how real S-LDSC analyses are actually run: LD is computed from a **reference panel** (e.g. 1000 Genomes), while the study's own single-cell data is only used to derive the specificity scores that define the annotation. So from here on we create annotations against 1000 Genomes plink files instead of `dd`'s genotypes, same functions, just swap `make_annot_from_donor_data` for `make_annot_from_bimfile`:

In [13]:
from cellink.resources import get_1000genomes_plink_files

plink_path, plink_prefix = get_1000genomes_plink_files(
    population="EUR",
)


def annotate_1000g(chrom_i, cell_type):
    annot_file = f"{cell_type.replace(' ', '_')}_1000G_{chrom_i}.annot.gz"
    if os.path.exists(annot_file):
        return
    make_annot_from_bimfile(
        bimfile=os.path.join(plink_path, f"{plink_prefix}{chrom_i}.bim"),
        annot_file=annot_file,
        gene_set_file=f"./ldsc_genesets/{cell_type.replace(' ', '_')}.GeneSet",
        gene_coord_file="gene_coords.txt",
        gene_coord_genome_build="GRCh37",
        bim_genome_build="GRCh37",  # 1000 Genomes Phase 3 plink files
        windowsize=100000,
        runner=runner,
    )


# the two cell types, and the Control set of all tested genes that Step 8 conditions them on
GENE_SETS = ["CD8 Naive", "CD4 Naive", "Control"]

# one make_annot.py run per chromosome and gene set, N_JOBS at a time
Parallel(n_jobs=N_JOBS, prefer="threads")(
    delayed(annotate_1000g)(chrom_i, gene_set) for chrom_i in range(1, 23) for gene_set in GENE_SETS
);

[2026-11-14 23:59:07,407] INFO:root: ~/cellink_data/1000genomes_plink_EUR/1000G_Phase3_plinkfiles.tgz already exists
[2026-11-14 23:59:07,409] WARNING:root: No checksum provided, skipping verification
[2026-11-14 23:59:07,490] INFO:cellink.tl.external._ldsc: Creating annotation file: /ldsc/make_annot.py --bimfile ~/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.2.bim --annot-file Control_1000G_2.annot.gz --gene-set-file ./ldsc_genesets/Control.GeneSet --gene-coord-file gene_coords.txt --windowsize 100000
[2026-11-14 23:59:07,492] INFO:cellink.tl.external._ldsc: Creating annotation file: /ldsc/make_annot.py --bimfile ~/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.3.bim --annot-file CD8_Naive_1000G_3.annot.gz --gene-set-file ./ldsc_genesets/CD8_Naive.GeneSet --gene-coord-file gene_coords.txt --windowsize 100000
[2026-11-14 23:59:07,492] INFO:cellink.tl.external._ldsc: Creating annotation file: /ldsc/make_annot.py --bimfile ~/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.2.bim --ann

**Continuous annotations (alternative)**: instead of thresholding to the top 10%, assign each SNP the actual specificity score of its nearest gene window. Use this if you'd rather not pick an arbitrary cutoff. Shown first from your own donor data (illustrative, same reasoning as above), then against the 1000G reference panel (the version compared with the binary annotation below; swap it into Step 5 to use continuous annotations):

In [14]:
for chrom_i in range(1, 23):
    for cell_type in ["CD8 Naive", "CD4 Naive"]:
        annot_file = f"{cell_type.replace(' ', '_')}_continuous_{chrom_i}.annot.gz"
        if os.path.exists(annot_file):
            continue
        dd_chrom = dd.sel(G_var=dd.G.var.chrom == str(chrom_i), C_var=dd.C.var.chrom == str(chrom_i)).copy()
        result = make_annot_from_donor_data(
            dd=dd_chrom,
            scores=specificity[cell_type],  # per-gene specificity scores (ENSG index)
            annot_file=annot_file,
            gene_coord_file="gene_coords_grch38.txt",
            gene_coord_genome_build="GRCh38",
            bim_genome_build="GRCh38",  # OneK1K genotypes
            windowsize=100_000,
            score_agg="max",  # SNP gets the max score of overlapping genes
        )
        print(
            f"chr{chrom_i} {cell_type}: {result['n_nonzero_snps']:,} non-zero SNPs, "
            f"{result['n_genes_matched']:,} genes matched"
        )

[2026-11-15 00:02:48,528] INFO:cellink.tl.external._ldsc: Exporting genotype data to PLINK format for annotation creation
[2026-11-15 00:02:48,602] INFO:cellink.io._export: Wrote PLINK 1 binary files ldsc_annot.bed/.bim/.fam (300 x 1)
[2026-11-15 00:02:48,920] INFO:cellink.tl.external._ldsc: Wrote continuous annotation: CD8_Naive_continuous_1.annot.gz (0 non-zero SNPs, 1656 genes matched)
[2026-11-15 00:02:48,922] INFO:cellink.tl.external._ldsc: Cleaned up: ldsc_annot.bim
[2026-11-15 00:02:48,923] INFO:cellink.tl.external._ldsc: Cleaned up: ldsc_annot.fam
[2026-11-15 00:02:48,925] INFO:cellink.tl.external._ldsc: Cleaned up: ldsc_annot.bed
chr1 CD8 Naive: 0 non-zero SNPs, 1,656 genes matched
[2026-11-15 00:02:49,130] INFO:cellink.tl.external._ldsc: Exporting genotype data to PLINK format for annotation creation
[2026-11-15 00:02:49,178] INFO:cellink.io._export: Wrote PLINK 1 binary files ldsc_annot.bed/.bim/.fam (300 x 1)
[2026-11-15 00:02:49,460] INFO:cellink.tl.external._ldsc: Wrote c

Against the 1000 Genomes reference panel (no `DonorData` needed), useful when running annotations independently from genotype data, and the version that's rsID-consistent with real GWAS:

In [15]:
for chrom_i in range(1, 23):
    for cell_type in ["CD8 Naive", "CD4 Naive"]:
        annot_file = f"{cell_type.replace(' ', '_')}_continuous_1000G_{chrom_i}.annot.gz"
        if os.path.exists(annot_file):
            continue
        result = make_annot_from_bimfile(
            bimfile=os.path.join(plink_path, f"{plink_prefix}{chrom_i}.bim"),
            scores=specificity[cell_type],
            annot_file=annot_file,
            gene_coord_file="gene_coords.txt",
            gene_coord_genome_build="GRCh37",
            bim_genome_build="GRCh37",  # 1000 Genomes Phase 3 plink files
            windowsize=100_000,
        )

[2026-11-15 00:03:20,368] INFO:cellink.tl.external._ldsc: Wrote continuous annotation: CD8_Naive_continuous_1000G_1.annot.gz (456830 non-zero SNPs, 1631 genes matched)
[2026-11-15 00:03:30,814] INFO:cellink.tl.external._ldsc: Wrote continuous annotation: CD4_Naive_continuous_1000G_1.annot.gz (493371 non-zero SNPs, 1631 genes matched)
[2026-11-15 00:03:41,488] INFO:cellink.tl.external._ldsc: Wrote continuous annotation: CD8_Naive_continuous_1000G_2.annot.gz (400686 non-zero SNPs, 1031 genes matched)
[2026-11-15 00:03:52,201] INFO:cellink.tl.external._ldsc: Wrote continuous annotation: CD4_Naive_continuous_1000G_2.annot.gz (437567 non-zero SNPs, 1031 genes matched)
[2026-11-15 00:04:01,041] INFO:cellink.tl.external._ldsc: Wrote continuous annotation: CD8_Naive_continuous_1000G_3.annot.gz (330593 non-zero SNPs, 891 genes matched)
[2026-11-15 00:04:09,943] INFO:cellink.tl.external._ldsc: Wrote continuous annotation: CD4_Naive_continuous_1000G_3.annot.gz (375068 non-zero SNPs, 891 genes mat

**Comparing binary vs continuous**: both files have identical columns; the only difference is the ANNOT distribution. Comparing the 1000G-based versions, since those are the ones used downstream:

In [16]:
binary_annot = pd.read_csv(f"CD8_Naive_1000G_{chrom}.annot.gz", sep="\t")
continuous_annot = pd.read_csv(f"CD8_Naive_continuous_1000G_{chrom}.annot.gz", sep="\t")

print("Columns are identical:", list(binary_annot.columns) == list(continuous_annot.columns))
print()
print("Binary ANNOT:")
print(binary_annot["ANNOT"].value_counts().sort_index())
print()
print("Continuous ANNOT (summary):")
print(continuous_annot["ANNOT"].describe())
print(f"  Non-zero SNPs: {(continuous_annot['ANNOT'] > 0).sum():,} / {len(continuous_annot):,}")

Columns are identical: True

Binary ANNOT:
ANNOT
0    108755
1     32368
Name: count, dtype: int64

Continuous ANNOT (summary):
count    141123.000000
mean          0.046782
std           0.094641
min           0.000000
25%           0.000000
50%           0.028144
75%           0.047088
max           0.728684
Name: ANNOT, dtype: float64
  Non-zero SNPs: 99,513 / 141,123


#### Step 5: Compute Cell-Type-Specific LD Scores

With annotations created, we compute LD scores incorporating cell-type-specific information, using the 1000G-based binary annotations (this tutorial's main path) via `compute_ld_scores_with_annotations_from_bimfile`. Swap in the `_continuous_1000G_` filenames to use the continuous ones instead; the call is otherwise identical. The Control set of all tested genes gets LD scores the same way.

In [17]:
def ld_scores(chrom_i, cell_type):
    out_prefix = f"cts_ldscores_{cell_type.replace(' ', '_')}.{chrom_i}"
    if os.path.exists(f"{out_prefix}.l2.ldscore.gz"):
        return
    compute_ld_scores_with_annotations_from_bimfile(
        bfile_prefix=os.path.join(plink_path, f"{plink_prefix}{chrom_i}"),
        annot_file=f"{cell_type.replace(' ', '_')}_1000G_{chrom_i}.annot.gz",
        out_prefix=out_prefix,
        run=True,
        runner=runner,
    )


# one ldsc run per chromosome and gene set, N_JOBS at a time (the step that takes longest)
Parallel(n_jobs=N_JOBS, prefer="threads")(
    delayed(ld_scores)(chrom_i, gene_set) for chrom_i in range(1, 23) for gene_set in GENE_SETS
);

[2026-11-15 00:07:27,708] INFO:cellink.tl.external._ldsc: Computing LD scores with annotations: /ldsc/ldsc.py --l2 --bfile ~/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.1 --annot CD4_Naive_1000G_1.annot.gz --out cts_ldscores_CD4_Naive.1 --ld-wind-cm 1.0 --yes-really
[2026-11-15 00:07:27,731] INFO:cellink.tl.external._ldsc: Computing LD scores with annotations: /ldsc/ldsc.py --l2 --bfile ~/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.2 --annot CD8_Naive_1000G_2.annot.gz --out cts_ldscores_CD8_Naive.2 --ld-wind-cm 1.0 --yes-really
[2026-11-15 00:07:27,731] INFO:cellink.tl.external._ldsc: Computing LD scores with annotations: /ldsc/ldsc.py --l2 --bfile ~/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.3 --annot CD4_Naive_1000G_3.annot.gz --out cts_ldscores_CD4_Naive.3 --ld-wind-cm 1.0 --yes-really
[2026-11-15 00:07:27,731] INFO:cellink.tl.external._ldsc: Computing LD scores with annotations: /ldsc/ldsc.py --l2 --bfile ~/cellink_data/1000genomes_plink_EUR/1000G.EUR.QC.2 --annot CD4_

#### Step 6: Prepare Reference LD Scores and Weights
For the final analysis, we need baseline LD scores and regression weights. These control for genomic confounders and ensure proper statistical inference. We download these from the 1000 Genomes reference panel, the same real, rsID-based files used in published S-LDSC analyses, so they merge correctly with the real GWAS downloaded in Step 7.

In [18]:
ldscores_path, ldscores_prefix = get_1000genomes_ld_scores(
    population="EUR",
    return_path=True,
)
ldweights_path, ldweights_prefix = get_1000genomes_ld_weights(
    population="EUR",
    return_path=True,
)

[2026-11-15 00:35:26,620] INFO:root: ~/cellink_data/1000genomes_ld_scores_EUR/1000G_Phase3_baselineLD_v2.2_ldscores.tgz already exists
[2026-11-15 00:35:26,622] WARNING:root: No checksum provided, skipping verification
[2026-11-15 00:35:26,702] INFO:root: ~/cellink_data/1000genomes_ld_weights_EUR/1000G_Phase3_weights_hm3_no_MHC.tgz already exists
[2026-11-15 00:35:26,703] WARNING:root: No checksum provided, skipping verification


Create the `.ldcts` file that lists the cell types: one line per cell type, with its LD score prefix and, after a comma, the prefix of the Control set of all tested genes. ldsc then estimates each cell type's contribution conditional on the baseline model and on all genes, as LDSC-SEG does ([Finucane et al. 2018](https://www.nature.com/articles/s41588-018-0081-4)). Without the control, a cell type's gene set would also pick up the general enrichment of heritability near genes, which is not specific to that cell type.

In [19]:
with open("celltype_ldscores.txt", "w") as f:
    for cell_type in ["CD8_Naive", "CD4_Naive"]:
        f.write(f"{cell_type}\tcts_ldscores_{cell_type}.,cts_ldscores_Control.\n")

#### Step 7: Prepare GWAS Summary Statistics
We download real GWAS summary statistics from the GWAS Catalog and munge them into LDSC's expected format. We reuse this same munged file later in the "Standard LDSC Analyses" section below, so it only needs to be downloaded and munged once.

In [20]:
gwas_summary_statistic_path_1 = get_gwas_catalog_study_summary_stats("GCST004787", return_path=True)
print(gwas_summary_statistic_path_1)

munged_file_1 = munge_sumstats(
    sumstats_file=gwas_summary_statistic_path_1,
    out_prefix=str(Path(Path(gwas_summary_statistic_path_1).stem).stem + "_munged"),
    info_min=0.9,
    maf_min=0.01,
    signed_sumstats=("logor", 0),
    run=True,
    p_col="p_value",
    snp_col="variant_id",
    a1_col="effect_allele",
    a2_col="other_allele",
    n_samples=336924,
    runner=runner,
)

[2026-11-15 00:35:26,817] INFO:root: Using cached summary statistics ~/cellink_data/gwas_catalog_summary_stats/GCST004787/28714975-GCST004787-EFO_0001645.h.tsv.gz (downloaded from http://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/GCST004001-GCST005000/GCST004787/harmonised/28714975-GCST004787-EFO_0001645.h.tsv.gz, build GRCh38)
~/cellink_data/gwas_catalog_summary_stats/GCST004787/28714975-GCST004787-EFO_0001645.h.tsv.gz
[2026-11-15 00:35:44,300] WARNING:cellink.tl.external._ldsc: munge_sumstats: dropping 8 all-missing column(s) ['hm_beta', 'hm_odds_ratio', 'hm_ci_lower', 'hm_ci_upper', 'ci_upper', 'odds_ratio', 'beta', 'ci_lower'] from '~/cellink_data/gwas_catalog_summary_stats/GCST004787/28714975-GCST004787-EFO_0001645.h.tsv.gz' before munging. munge_sumstats.py's blanket dropna(how='any') would otherwise drop every row because of these, zeroing out the entire result ('No objects to concatenate'). Writing sanitized copy to '28714975-GCST004787-EFO_0001645.h_munged.sanitized_i

#### Step 8: Run Cell-Type-Specific Heritability Analysis
Finally, we run the cell-type-specific heritability analysis, which tests whether each cell type shows enrichment for trait heritability. `ref_ld_chr`/`w_ld_chr` use the real 1000G baseline/weights downloaded in Step 6, the same rsID convention as the GWAS sumstats and the cell-type-specific LD scores, so the merge step finds real overlap.

In [21]:
result = estimate_celltype_specific_heritability(
    sumstats_file=str(Path(Path(gwas_summary_statistic_path_1).stem).stem + "_munged.sumstats.gz"),
    ref_ld_chr=os.path.join(ldscores_path, ldscores_prefix),
    w_ld_chr=os.path.join(ldweights_path, ldweights_prefix),
    ref_ld_chr_cts="celltype_ldscores.txt",
    out_prefix="CHD_CD8_Naive_h2",
    run=True,
    runner=runner,
)

[2026-11-15 00:39:31,414] INFO:cellink.tl.external._ldsc: Running cell-type-specific heritability analysis: /ldsc/ldsc.py --h2-cts 28714975-GCST004787-EFO_0001645.h_munged.sumstats.gz --ref-ld-chr ~/cellink_data/1000genomes_ld_scores_EUR/baselineLD. --w-ld-chr ~/cellink_data/1000genomes_ld_weights_EUR/weights.hm3_noMHC. --ref-ld-chr-cts celltype_ldscores.txt --out CHD_CD8_Naive_h2
[2026-11-15 00:39:31,420] INFO:cellink.tl._runner: Executing: singularity exec -B .:/data -B ~/cellink_data:/cellink_data --pwd /data ldsc.sif /ldsc/ldsc.py --h2-cts /data/28714975-GCST004787-EFO_0001645.h_munged.sumstats.gz --ref-ld-chr /cellink_data/1000genomes_ld_scores_EUR/baselineLD. --w-ld-chr /cellink_data/1000genomes_ld_weights_EUR/weights.hm3_noMHC. --ref-ld-chr-cts /data/celltype_ldscores.txt --out /data/CHD_CD8_Naive_h2
[2026-11-15 00:41:59,392] INFO:cellink.tl._runner: *********************************************************************
* LD Score Regression (LDSC)
* Version 1.0.1
* (C) 2014-2019

### Standard LDSC Analyses
Beyond cell-type-specific analysis, LDSC can also be used for standard heritability estimation and genetic correlation analysis. Here we demonstrate these capabilities, reusing the GWAS summary statistics downloaded and munged in Step 7.


In [22]:
runner = configure_ldsc_runner("ldsc_singularity")

Download reference LD scores and weights for heritability estimation:

In [23]:
ldscores_path, ldscores_prefix = get_1000genomes_ld_scores(
    population="EUR",
    return_path=True,
)
ldweights_path, ldweights_prefix = get_1000genomes_ld_weights(
    population="EUR",
    return_path=True,
)

[2026-11-15 00:41:59,483] INFO:root: ~/cellink_data/1000genomes_ld_scores_EUR/1000G_Phase3_baselineLD_v2.2_ldscores.tgz already exists
[2026-11-15 00:41:59,484] WARNING:root: No checksum provided, skipping verification
[2026-11-15 00:41:59,500] INFO:root: ~/cellink_data/1000genomes_ld_weights_EUR/1000G_Phase3_weights_hm3_no_MHC.tgz already exists
[2026-11-15 00:41:59,501] WARNING:root: No checksum provided, skipping verification


Now estimate SNP heritability:

In [24]:
h2_results = estimate_heritability(
    sumstats_file=str(Path(Path(gwas_summary_statistic_path_1).stem).stem + "_munged.sumstats.gz"),
    ref_ld_chr=os.path.join(ldscores_path, ldscores_prefix),
    w_ld_chr=os.path.join(ldweights_path, ldweights_prefix),
    out_prefix=str(Path(Path(gwas_summary_statistic_path_1).stem).stem + "_h2"),
    run=True,
    runner=runner,
)

[2026-11-15 00:41:59,510] INFO:cellink.tl.external._ldsc: Estimating heritability: /ldsc/ldsc.py --h2 28714975-GCST004787-EFO_0001645.h_munged.sumstats.gz --ref-ld-chr ~/cellink_data/1000genomes_ld_scores_EUR/baselineLD. --w-ld-chr ~/cellink_data/1000genomes_ld_weights_EUR/weights.hm3_noMHC. --out 28714975-GCST004787-EFO_0001645.h_h2
[2026-11-15 00:41:59,513] INFO:cellink.tl._runner: Executing: singularity exec -B .:/data -B ~/cellink_data:/cellink_data --pwd /data ldsc.sif /ldsc/ldsc.py --h2 /data/28714975-GCST004787-EFO_0001645.h_munged.sumstats.gz --ref-ld-chr /cellink_data/1000genomes_ld_scores_EUR/baselineLD. --w-ld-chr /cellink_data/1000genomes_ld_weights_EUR/weights.hm3_noMHC. --out /data/28714975-GCST004787-EFO_0001645.h_h2
[2026-11-15 00:42:51,553] INFO:cellink.tl._runner: *********************************************************************
* LD Score Regression (LDSC)
* Version 1.0.1
* (C) 2014-2019 Brendan Bulik-Sullivan and Hilary Finucane
* Broad Institute of MIT and Harv

#### Genetic Correlation Analysis
Genetic correlation analysis quantifies the shared genetic basis between two traits. We download a second GWAS for coronary artery disease and compute the genetic correlation.

In [25]:
gwas_summary_statistic_path_2 = get_gwas_catalog_study_summary_stats("GCST90043957", return_path=True)
gwas_summary_statistic_path_2

[2026-11-15 00:42:51,634] INFO:root: Using cached summary statistics ~/cellink_data/gwas_catalog_summary_stats/GCST90043957/34737426-GCST90043957-MONDO_0021661.h.tsv.gz (downloaded from http://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/GCST90043001-GCST90044000/GCST90043957/harmonised/34737426-GCST90043957-MONDO_0021661.h.tsv.gz, build GRCh38)


PosixPath('~/cellink_data/gwas_catalog_summary_stats/GCST90043957/34737426-GCST90043957-MONDO_0021661.h.tsv.gz')

In [26]:
munged_file_2 = munge_sumstats(
    sumstats_file=gwas_summary_statistic_path_2,
    out_prefix=str(Path(Path(gwas_summary_statistic_path_2).stem).stem + "_munged"),
    info_min=0.9,
    maf_min=0.01,
    signed_sumstats=("beta", 0),
    run=True,
    p_col="p_value",
    snp_col="variant_id",
    a1_col="effect_allele",
    a2_col="other_allele",
    n_samples=456348,
    runner=runner,
)

[2026-11-15 00:43:11,276] WARNING:cellink.tl.external._ldsc: munge_sumstats: dropping 6 all-missing column(s) ['hm_odds_ratio', 'hm_ci_lower', 'hm_ci_upper', 'odds_ratio', 'ci_lower', 'ci_upper'] from '~/cellink_data/gwas_catalog_summary_stats/GCST90043957/34737426-GCST90043957-MONDO_0021661.h.tsv.gz' before munging. munge_sumstats.py's blanket dropna(how='any') would otherwise drop every row because of these, zeroing out the entire result ('No objects to concatenate'). Writing sanitized copy to '34737426-GCST90043957-MONDO_0021661.h_munged.sanitized_input.tsv'. Pass drop_allna_columns=False to disable.
[2026-11-15 00:45:39,958] INFO:cellink.tl.external._ldsc: Running munge_sumstats: /ldsc/munge_sumstats.py --sumstats 34737426-GCST90043957-MONDO_0021661.h_munged.sanitized_input.tsv --out 34737426-GCST90043957-MONDO_0021661.h_munged --N 456348 --signed-sumstats beta,0 --p p_value --a1 effect_allele --a2 other_allele --snp variant_id
[2026-11-15 00:45:39,965] INFO:cellink.tl._runner: Exe

Finally, estimate the genetic correlation between the two traits:

In [27]:
rg_results = estimate_genetic_correlation(
    sumstats_files=[
        str(Path(Path(gwas_summary_statistic_path_1).stem).stem + "_munged.sumstats.gz"),
        str(Path(Path(gwas_summary_statistic_path_2).stem).stem + "_munged.sumstats.gz"),
    ],
    ref_ld_chr=os.path.join(ldscores_path, ldscores_prefix),
    w_ld_chr=os.path.join(ldweights_path, ldweights_prefix),
    out_prefix="CHD_rg",
    run=True,
    runner=runner,
)

[2026-11-15 00:48:56,275] INFO:cellink.tl.external._ldsc: Estimating genetic correlation: /ldsc/ldsc.py --rg 28714975-GCST004787-EFO_0001645.h_munged.sumstats.gz,34737426-GCST90043957-MONDO_0021661.h_munged.sumstats.gz --ref-ld-chr ~/cellink_data/1000genomes_ld_scores_EUR/baselineLD. --w-ld-chr ~/cellink_data/1000genomes_ld_weights_EUR/weights.hm3_noMHC. --out CHD_rg
[2026-11-15 00:48:56,279] INFO:cellink.tl._runner: Executing: singularity exec -B .:/data -B ~/cellink_data:/cellink_data --pwd /data ldsc.sif /ldsc/ldsc.py --rg 28714975-GCST004787-EFO_0001645.h_munged.sumstats.gz,34737426-GCST90043957-MONDO_0021661.h_munged.sumstats.gz --ref-ld-chr /cellink_data/1000genomes_ld_scores_EUR/baselineLD. --w-ld-chr /cellink_data/1000genomes_ld_weights_EUR/weights.hm3_noMHC. --out /data/CHD_rg
[2026-11-15 00:50:52,351] INFO:cellink.tl._runner: *********************************************************************
* LD Score Regression (LDSC)
* Version 1.0.1
* (C) 2014-2019 Brendan Bulik-Sulliva

# Summary
This tutorial demonstrated how to perform comprehensive LDSC analyses using the `cellink` package, including:

1. Cell-type-specific heritability analysis: Identifying which cell types are most relevant to complex traits
2. SNP heritability estimation: Quantifying the proportion of trait variance explained by common genetic variants
3. Genetic correlation analysis: Measuring shared genetic architecture between traits

The `cellink` package simplifies these analyses by providing unified wrapper functions that handle data formatting, file management, and command execution for LDSC and its auxiliary tools.